In [9]:
import numpy as np
import re
import pandas as pd
import tqdm

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import Pauli, DensityMatrix
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, amplitude_damping_error, PauliError, phase_damping_error

from scipy.optimize import curve_fit
from scipy.optimize import least_squares

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
df = pd.read_csv('./data/sk-hamiltonians.csv')

def fold_circuit(circuit: QuantumCircuit, scale_factor: float, folding_method='gate') -> QuantumCircuit:
    if scale_factor < 1 or int(scale_factor) % 2 == 0:
        warnings.warn(f"Scale factor {scale_factor} adjusted to nearest odd integer ≥ 1")
        scale_factor = max(1, 2 * int(np.ceil((scale_factor - 1) / 2)) + 1)
    
    scale_int = int(scale_factor)
    
    if scale_int == 1:
        copied_circuit = circuit.copy()
        copied_circuit.remove_final_measurements()
        return copied_circuit
    
    # Remove measurements for folding
    circuit_no_measure = circuit.copy()
    circuit_no_measure.remove_final_measurements()
    
    number_of_folds = (scale_int - 1) // 2
    
    # Create folded circuit
    folded_circuit = QuantumCircuit(circuit.num_qubits)

    if folding_method=='gate':
        # Get gates to fold (excluding barriers)
        gates_to_fold = []
        for instruction in circuit_no_measure.data:
            if instruction.operation.name not in ['barrier']:
                gates_to_fold.append(instruction)
    
        for instruction in gates_to_fold:
            folded_circuit.append(instruction.operation, instruction.qubits)
            
            for _ in range(number_of_folds):
                folded_circuit.append(instruction.operation, instruction.qubits)
                folded_circuit.append(instruction.operation.inverse(), instruction.qubits)

    elif folding_method=='circuit':
        # Apply original circuit
        folded_circuit.compose(circuit_no_measure, inplace=True)
        
        # Apply folding pairs (circuit + inverse circuit)
        for _ in range(number_of_folds):
            folded_circuit.compose(circuit_no_measure, inplace=True)
            folded_circuit.compose(circuit_no_measure.inverse(), inplace=True)
    else:
        raise ValueError("Unknown folding method. Possible values are 'gate' and 'circuit'.")
    
    folded_circuit.metadata = {'scale_factor': scale_factor, 'folding_method': folding_method}
    
    return folded_circuit

def residuals(params, x, y):
    A, B, C = params
    return A * np.exp(B * x) + C - y

def exponential_extrapolation(scales, values):
    x = np.array(scales)
    y = np.array(values)
    res = least_squares(residuals, x0=[1, -0.1, 0], args=(x, y))
    A, B, C = res.x
    return A + C

def gate_folding_zne(circuit, backend_noisy):
    noisy_gate = []
    noisy_circuit = transpile(circuit, backend_noisy)
    noisy_circuit.save_density_matrix()

    result = backend_noisy.run(noisy_circuit).result()
    dm = DensityMatrix(result.data(0)['density_matrix'])
    expectations = np.array([np.real(dm.expectation_value(obs)) for obs in observables])
    e_noisy = np.sum(expectations * coefs)
    noisy_gate.append(e_noisy)
    noisy_circuit = transpile(circuit, backend_noisy)
    fold_gate_3 = fold_circuit(noisy_circuit, 3)
    fold_gate_3.save_density_matrix()

    result = backend_noisy.run(fold_gate_3).result()
    dm = DensityMatrix(result.data(0)['density_matrix'])
    expectations = np.array([np.real(dm.expectation_value(obs)) for obs in observables])
    e_noisy_gate_3 = np.sum(expectations * coefs)
    noisy_gate.append(e_noisy_gate_3)
    noisy_circuit = transpile(circuit, backend_noisy)
    fold_gate_5 = fold_circuit(noisy_circuit, 5)
    fold_gate_5.save_density_matrix()

    result = backend_noisy.run(fold_gate_5).result()
    dm = DensityMatrix(result.data(0)['density_matrix'])
    expectations = np.array([np.real(dm.expectation_value(obs)) for obs in observables])
    e_noisy_gate_5 = np.sum(expectations * coefs)
    noisy_gate.append(e_noisy_gate_5)

    exp = exponential_extrapolation([1., 3., 5.], noisy_gate)
    return exp

def build_noise_model(model, two_qubit_error_rate):
    noise_model = NoiseModel()
    if model == 'depolarization':
        cnot_error = depolarizing_error(two_qubit_error_rate, 2)
        noise_model.add_all_qubit_quantum_error(cnot_error, ['cx'])
            
    elif model == 'pauli':
        p_id = 1 - two_qubit_error_rate/2
        p_x = two_qubit_error_rate/6
        p_y = p_x - p_x/3
        p_z = p_x + p_x/3
        pauli_strings = ["II", "IX", "XI", "YI", "IY", "ZI", "IZ", "XX", "YY", "ZZ", "XY", "YX", "XZ", "ZX", "YZ", "ZY"]
        probas = [p_id**2, p_id*p_x, p_id*p_x, p_id*p_y, p_id*p_y, p_id*p_z, p_id*p_z, p_x**2, p_y**2, p_z**2, p_x*p_y, p_x*p_y, p_x*p_z, p_x*p_z, p_y*p_z, p_y*p_z]
        pauli_error = PauliError(pauli_strings, probas)
        noise_model.add_all_qubit_quantum_error(pauli_error, ['cx'])
            
    elif model == 'composite':
        amp_err = amplitude_damping_error(two_qubit_error_rate / 2)
        phase_err = phase_damping_error(two_qubit_error_rate / 2)
        two_qubit_amp = amp_err.tensor(amp_err)
        two_qubit_phase = phase_err.tensor(phase_err)
        dep_err = depolarizing_error(two_qubit_error_rate, 2)
        combined_error = two_qubit_amp.compose(two_qubit_phase).compose(dep_err)
        noise_model.add_all_qubit_quantum_error(combined_error, 'cx')
        
        # Use density matrix simulator for noisy simulation
    simulator = AerSimulator(method='density_matrix', noise_model=noise_model)
    return simulator

In [2]:
num_qubits = 12
num_layers = 4

ansatz = TwoLocal(
    num_qubits,
    rotation_blocks='ry',
    entanglement_blocks='cx',
    entanglement='circular',
    reps=num_layers,
    skip_final_rotation_layer=True,
    insert_barriers=True
)

# =========================
# Observables
# =========================
observables = []

for i in range(num_qubits):
    for j in range(i+1, num_qubits):
        pauli = ['I'] * num_qubits
        pauli[i] = 'Z'
        pauli[j] = 'Z'
        observables.append(Pauli(''.join(pauli)))

for i in range(num_qubits):
    pauli = ['I'] * num_qubits
    pauli[i] = 'X'
    observables.append(Pauli(''.join(pauli)))

In [15]:
results = {
    "zne": [],
}
model = 'composite'
two_qubit_error_rate = 0.1
backend_noisy = build_noise_model(model, two_qubit_error_rate)

for i in tqdm.tqdm(range(len(df))):
    params_str = df['params'][i]
    params_clean = re.sub(r'\s+', ' ', params_str.strip('[]\n '))
    params = np.fromstring(params_clean, sep=' ')

    # coefs
    coefs = np.array(eval(df['coefs'][i]), dtype=float)
    coefs = np.concatenate((coefs, np.ones(num_qubits)))

    circuit = ansatz.assign_parameters(params)

    gate_exp = gate_folding_zne(circuit, backend_noisy)
    results["zne"].append(
        gate_exp
    )

df_out = df.copy()
for key in results:
    df_out[key] = results[key]

df_out.to_csv("./data/error_stats/zne_" + model + str(two_qubit_error_rate).replace('.', '') + ".csv", index=False)

100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [06:23<00:00,  3.83s/it]
